In [14]:
import osmnx as ox

from genesis.models import Environment

In [2]:
G = ox.load_graphml("tests/data/test_rng.ml")
E = Environment(G)

In [16]:
isinstance(E, Environment)   #=='genesis.models.Environment'

True

In [6]:
import pandas as pd
import numpy as np

d = {1:1, 2:4, 3:6, 4:2}
s = pd.Series(d)

print(max(s))
print(np.mean(s))
print(np.median(s))

6
3.25
3.0


In [5]:
ss=0
for v in d.values():
    ss+=v
print(ss/len(d))

3.25


In [7]:
import this

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!


In [1]:
import logging

In [2]:
logging.debug("A DEBUG Message")
logging.info("An INFO")
logging.warning("A WARNING")
logging.error("An ERROR")
logging.critical("A message of CRITICAL severity")

ERROR:root:An ERROR
CRITICAL:root:A message of CRITICAL severity


In [4]:
logging.basicConfig(level=logging.INFO, filename="py_log.log",filemode="w")
logging.debug("A DEBUG Message")
logging.info("An INFO")
logging.warning("A WARNING")
logging.error("An ERROR")
logging.critical("A message of CRITICAL severity")

ERROR:root:An ERROR
CRITICAL:root:A message of CRITICAL severity


In [2]:
max_time =17.2

print(f"{max_time=}")

max_time=17.2


In [1]:
from genesis.models import Environment
from genesis.calculators import Metrics
from genesis.swiss_knife import ssfpl
import networkx as nx
import osmnx as ox
import logging as lg

def test_calc_ip10():
    E = Environment(ox.load_graphml("tests/data/test_rng.ml"))
    start_node = list(E.G.nodes())[100]
    lngs = nx.single_source_dijkstra_path_length(E.G, start_node, weight='length')
    covered = [1 if t<=10 else 0 for t in lngs.values()]
    ip_real = round(100*sum(covered)/len(covered), 2)

    ip_test = Metrics.calc_metric(E, 
                                   start_node, 
                                   path_function=ssfpl,
                                   metric_function=Metrics.calc_ip,
                                   weight='length')
    lg.basicConfig(level=lg.DEBUG, filename="test_metrics.log", filemode="w")
    # lg.debug(f"{metric_function}: {val=}")
    lg.debug("тестова запись")
    assert ip_real==ip_test

In [6]:
from genesis.calculators import Metrics
from genesis.swiss_knife import ssfpl
from genesis.models import Environment
import numpy as np
import networkx as nx


def create_E():
    G = nx.MultiDiGraph()
    G.add_edge(1, 2, key=0, travel_time=3)
    G.add_edge(1, 3, key=0, travel_time=2)
    G.add_edge(1, 4, key=0, travel_time=5)
    G.add_edge(2, 3, key=0, travel_time=2)
    G.add_edge(2, 5, key=0, travel_time=5)
    G.add_edge(2, 6, key=0, travel_time=7)
    G.add_edge(3, 2, key=0, travel_time=3)
    G.add_edge(3, 7, key=0, travel_time=3)
    G.add_edge(3, 8, key=0, travel_time=4)
    G.add_edge(3, 9, key=0, travel_time=5)
    G.add_edge(4, 3, key=0, travel_time=2)
    G.add_edge(4, 8, key=0, travel_time=4)
    G.add_edge(4, 10, key=0, travel_time=12)
    G.add_edge(4, 11, key=0, travel_time=10)
    G.add_edge(5, 8, key=0, travel_time=8)
    G.add_edge(5, 10, key=0, travel_time=8)
    G.add_edge(5, 12, key=0, travel_time=6)
    G.add_edge(5, 13, key=0, travel_time=9)
    G.add_edge(5, 12, key=0, travel_time=8)
    G.add_edge(6, 8, key=0, travel_time=1)
    G.add_edge(6, 9, key=0, travel_time=4)
    G.add_edge(6, 12, key=0, travel_time=4)
    G.add_edge(7, 10, key=0, travel_time=4)
    G.add_edge(7, 11, key=0, travel_time=9)
    G.add_edge(7, 13, key=0, travel_time=7)
    G.add_edge(8, 10, key=0, travel_time=6)
    G.add_edge(8, 13, key=0, travel_time=8)
    G.add_edge(9, 14, key=0, travel_time=10)
    G.add_edge(10, 13, key=0, travel_time=7)
    G.add_edge(10, 14, key=0, travel_time=5)
    G.add_edge(11, 12, key=0, travel_time=7)
    G.add_edge(11, 15, key=0, travel_time=7)
    G.add_edge(12, 13, key=0, travel_time=8)
    return Environment(G)

E = create_E()
print("max:", Metrics.calc_metric(E, 1, ssfpl, np.max))
print("mean:", Metrics.calc_metric(E, 1, ssfpl, np.mean))
print("median:", Metrics.calc_metric(E, 1, ssfpl, np.median))
print("ip-10:", Metrics.calc_metric(E, 1, ssfpl, Metrics.calc_ip()))
print("ip-20:", Metrics.calc_metric(E, 1, ssfpl, Metrics.calc_ip(ip_val=20)))

max: 21
mean: 8.67
median: 8.0
ip-10: 66.67
ip-20: 93.33


In [1]:
import osmnx as ox
import networkx as nx
import numpy as np
import pandas as pd

from genesis.models import Environment
from genesis.calculators import Metrics
from genesis.swiss_knife import ssfpl

E = Environment(ox.load_graphml("tests/data/test_rng.ml"))

In [6]:
def calc_metric(E:Environment, 
                    node:int, 
                    # path_function, 
                    # metric_function, 
                    # weight: str = "travel_time",
                    # precision: int = 2,
                    **kwargs):
        route_lens = nx.single_source_dijkstra_path_length(E.G, node, weight='length')
        less_500 = [1 if d<500 else 0 for d in route_lens.values()]
        return route_lens, sum(less_500)/len(less_500)

start_node = list(E.G.nodes())[2000]
route_lens, cm = calc_metric(E, start_node)
# assert calc_metric(E, start_node)==100

In [8]:
cm

0.026344476744186048

In [7]:
route_lens

{2042076750: 0,
 2042076753: 6.191,
 2042076746: 26.895,
 2042076766: 34.807,
 2042076739: 42.406,
 2042076737: 52.309,
 2042076744: 60.144999999999996,
 2042076736: 78.493,
 2042076727: 85.503,
 2042076725: 103.413,
 2042076794: 104.218,
 2042076745: 104.401,
 2042076763: 104.477,
 2042076717: 116.721,
 2042076733: 119.19399999999999,
 2042076760: 130.272,
 2042076713: 134.926,
 2042076823: 135.591,
 2042076724: 139.927,
 2042076743: 146.817,
 2042076703: 155.272,
 2042076731: 158.884,
 2042076762: 170.724,
 2042076792: 175.65,
 2042076720: 176.99699999999999,
 2042076812: 177.79000000000002,
 2042076694: 180.11899999999997,
 2042076847: 180.28300000000002,
 2042076806: 182.70800000000003,
 2042076795: 185.211,
 2042076732: 188.73999999999998,
 2042076849: 190.27900000000002,
 2042076804: 192.43100000000004,
 2042076701: 194.053,
 2042076690: 194.39299999999997,
 2042076741: 197.798,
 2042076803: 203.77100000000004,
 2042076860: 210.08200000000002,
 2042076674: 215.97599999999997,
 20